# 01 - Data and features
Works on **Kaggle notebooks or Colab**. Sources:
- `malicious_phish.csv` (Kaggle sid321axn): **main training pool**
- `balanced_urls.csv` (Kaggle samahsadiq): **external test source** (never trained on)
- URLhaus CSV dump: **fresh malicious URLs** (recall-only test)

Kaggle: add the two datasets via *Add Data*, upload `url-guard.zip` and the URLhaus file as a private dataset, and turn Internet **on** (needed for `pip install tldextract`).
Colab: upload the zip and the three files to `/content`.

In [ ]:
import os, sys, glob, zipfile

base = "/kaggle/working" if os.path.isdir("/kaggle/working") else ("/content" if os.path.isdir("/content") else os.getcwd())

# Unpack zip if found
for r in [base, "/content", ".", "/kaggle/input"]:
    for pat in ["*url-guard*.zip", "url-guard.zip"]:
        hits = glob.glob(os.path.join(r, pat))
        if hits:
            z = sorted(hits, key=os.path.getmtime)[-1]
            print("Unpacking:", z)
            zipfile.ZipFile(z).extractall(base)
            break

# Ensure src is on sys.path
for d in [os.path.join(base, "url-guard", "src"), os.path.join(base, "src"), "src", "/content/url-guard/src"]:
    if os.path.isdir(d) and d not in sys.path:
        sys.path.insert(0, d)
        print("Added to path:", d)

if os.path.isdir(os.path.join(base, "url-guard")):
    os.chdir(os.path.join(base, "url-guard"))

print("project dir:", os.getcwd())


In [ ]:
%pip install -q tldextract pytest pyarrow
!python -m pytest -q tests

## Locate the three files (override the paths by hand if the guess is wrong)

In [ ]:
def locate(name, *alts):
    candidates = [name] + list(alts)
    for c in candidates:
        for prefix in ["/content", ".", "/kaggle/input", "data"]:
            p = os.path.join(prefix, c)
            if os.path.isfile(p):
                return p
        for prefix in ["/content", ".", "/kaggle/input"]:
            hits = glob.glob(os.path.join(prefix, "**", c), recursive=True)
            if hits:
                return hits[0]
    return None

MAIN = locate("malicious_phish.csv")
BAL  = locate("balanced_urls.csv")
UH   = locate("urls.csv", "csv.txt", "*urlhaus*", "csv_recent*")

for k, v in [("main (sid321axn)", MAIN), ("external (samahsadiq)", BAL), ("urlhaus dump", UH)]:
    print(f"{k:24s} {v}")

assert MAIN and BAL and UH, "Could not find all 3 files in /content. Please ensure malicious_phish.csv, balanced_urls.csv, and urls.csv are uploaded."
print("All 3 files successfully located!")


## Load raw sources
`newest_n` keeps only the newest URLhaus rows. The full dump is huge and would swamp the malicious class with IP-address malware.

In [ ]:
import pandas as pd
import data_loader as dl
main = dl.load_kaggle(MAIN)
bal  = dl.load_balanced_urls(BAL)
uh   = dl.load_urlhaus_csv(UH, newest_n=100_000)
for n, d in [("main", main), ("balanced", bal), ("urlhaus", uh)]:
    print(f"{n:9s} rows={len(d):>8,}  malicious={d.label.mean():.1%}")
print("\nmain subtypes:\n", main.subtype.value_counts())
print("\nurlhaus date range:", uh.dateadded.min(), "->", uh.dateadded.max())
uh.head(3)

## How independent are the sources?
If `balanced_urls.csv` is largely derived from the same origin as `malicious_phish.csv`, it is a weaker external test than it looks.

In [ ]:
dl.overlap_report(main, bal, "main", "balanced")

## Merge and clean
Order matters: a URL present in several sources stays with the **first** one, so the external set is URL-disjoint from main.

In [ ]:
os.makedirs("data/processed", exist_ok=True)
df = dl.build_dataset([main, bal, uh], save_to="data/processed/urls_merged.csv")
df.groupby(["source", "subtype"]).size().to_frame("rows")

## Source-bias check
Can features alone tell which **source** a URL came from, within the same class? Look at the `gap` column. Large gaps mean the model can learn source artifacts instead of maliciousness.

In [ ]:
bias = dl.source_bias_check(df)

## What does URLhaus actually contain?
If nearly all of it is IP-address hosts, then a high recall on it mostly shows the model learned the IP-host rule.

In [ ]:
from features import extract_features_df
u = df[df.source == "urlhaus"].sample(min(20000, (df.source == "urlhaus").sum()), random_state=0)
Xu = extract_features_df(u["url"])
print("share with IP host:", round(Xu.has_ip_host.mean(), 3), "| share with non-default port:", round(Xu.has_port.mean(), 3))

## Feature matrix and a first look

In [ ]:
X = extract_features_df(df["url"])
print(X.shape, "| rows with failed extraction:", int(X.isna().any(axis=1).sum()))
view = X.assign(label=df["label"].values, source=df["source"].values)
view.groupby(["source", "label"])[["url_length", "n_special", "n_subdomains", "keyword_count", "min_brand_dist", "has_ip_host"]].mean().round(2)

In [ ]:
out = pd.concat([df.reset_index(drop=True), X], axis=1).dropna(subset=list(X.columns))
out.to_parquet("data/processed/urls_features.parquet", index=False)
print("saved", out.shape, "->", os.path.abspath("data/processed/urls_features.parquet"))